# 洪水事件体积-时长标度分析（Notebook 版）

本笔记本对**已提取好的洪水事件**（上一阶段输出的 `*_flood_events_timeseries_calendar.csv` 和 `per-year/*_timeseries.csv`）进行：
1. **逐年最大累积体积**计算（对每个年内事件时序，计算窗口 \(T=1..L\) 天的滑动和最大值），输出 `max_volume_results.csv`；
2. **标度关系**拟合：在 \(\log T\)–\(\log V_{max}(T)\) 上线性回归，输出 `scaling_results.csv`；
3. 自动汇总为 `ALL_max_volume_results.csv` 与 `ALL_scaling_results.csv`。

> **使用说明**：先在下个单元修改 `INPUT_ROOT`（指向你“分站点输出”的根目录，如 `.../grdc_out_calendar`）和 `OUTPUT_ROOT`，然后依次运行所有单元。

In [1]:
from pathlib import Path
INPUT_ROOT = r"D:\workroom\GPLW\makeUP\flood"
OUTPUT_ROOT = r"grdc_scaling_out"
INPUT_ROOT, OUTPUT_ROOT

('D:\\workroom\\GPLW\\makeUP\\flood', 'grdc_scaling_out')

In [2]:
import re, numpy as np, pandas as pd
from pathlib import Path
from typing import List, Tuple
try:
    from scipy.stats import linregress as _scipy_linregress
    def linregress(x, y):
        lr = _scipy_linregress(x, y)
        return lr.slope, lr.intercept, lr.rvalue, lr.pvalue, lr.stderr
except Exception:
    def linregress(x, y):
        x = np.asarray(x, dtype=float); y = np.asarray(y, dtype=float)
        A = np.vstack([x, np.ones_like(x)]).T
        slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
        r = np.corrcoef(x, y)[0,1] if x.size>1 else 0.0
        return float(slope), float(intercept), float(r), np.nan, np.nan
print('Deps ready.')

Deps ready.


In [3]:
def find_station_timeseries_csv(station_dir: Path) -> Tuple[str, object]:
    if not station_dir.is_dir():
        return 'none', None
    cands = list(station_dir.glob('*_flood_events_timeseries_calendar.csv'))
    if cands:
        return 'events_timeseries', cands[0]
    pery = station_dir / 'per-year'
    frags = sorted(pery.glob('*_flood_*_timeseries.csv')) if pery.exists() else []
    if frags:
        return 'fragments', frags
    return 'none', None
print('Finder ready.')

Finder ready.


In [4]:
def load_station_event_series_events_timeseries(ts_path: Path) -> pd.DataFrame:
    df = pd.read_csv(ts_path, parse_dates=['time'])
    if 'runoff' not in df.columns:
        raise ValueError(f"{ts_path} 缺少 'runoff' 列")
    use_cols = ['time','runoff']
    if 'station_name' in df.columns:
        use_cols.append('station_name')
    out = df[use_cols].dropna(subset=['time']).copy()
    out['year'] = out['time'].dt.year
    out = out.sort_values('time').reset_index(drop=True)
    if 'station_name' not in out.columns:
        out['station_name'] = ts_path.parent.name
    return out

def load_station_event_series_from_fragments(frags: List[Path]) -> pd.DataFrame:
    dfs = []
    for p in frags:
        df = pd.read_csv(p, parse_dates=['time'])
        cols = {c.lower(): c for c in df.columns}
        tcol = cols.get('time'); rcol = cols.get('runoff')
        if tcol is None or rcol is None:
            raise ValueError(f"{p} 缺少 'time'/'runoff' 列")
        sub = df[[tcol, rcol]].rename(columns={tcol:'time', rcol:'runoff'}).dropna(subset=['time'])
        sub['station_name'] = p.parent.parent.name
        dfs.append(sub)
    if not dfs:
        return pd.DataFrame(columns=['time','runoff','station_name','year'])
    out = pd.concat(dfs, ignore_index=True)
    out['year'] = pd.to_datetime(out['time']).dt.year
    out = out.sort_values('time').reset_index(drop=True)
    return out
print('Loaders ready.')

Loaders ready.


In [5]:
def analyze_flood_scaling_for_station_noresample(station_dir: Path):
    mode, payload = find_station_timeseries_csv(station_dir)
    if mode == 'none':
        return None, None, station_dir.name
    if mode == 'events_timeseries':
        df = load_station_event_series_events_timeseries(payload)
    else:
        df = load_station_event_series_from_fragments(payload)
    if df.empty:
        return None, None, station_dir.name
    st_name = df['station_name'].dropna().unique()
    station_name = st_name[0] if len(st_name) else station_dir.name
    results, slopes = [], []
    for year, g in df.groupby('year'):
        g = g.sort_values('time')
        series = g['runoff'].astype(float).reset_index(drop=True)
        L = len(series)
        if L < 2: continue
        max_volumes = {}
        for T in range(1, L+1):
            mv = series.rolling(window=T, min_periods=T).sum().max()
            max_volumes[T] = float(mv) if pd.notna(mv) else 0.0
        row = {'Year': int(year)}; row.update(max_volumes); results.append(row)
        xs = np.array([t for t, v in max_volumes.items() if v > 0], dtype=float)
        ys = np.array([v for t, v in max_volumes.items() if v > 0], dtype=float)
        if xs.size >= 2 and ys.size >= 2:
            slope, intercept, r_value, p_value, std_err = linregress(np.log(xs), np.log(ys))
            slopes.append({'Year': int(year), 'Slope': slope, 'Intercept': intercept, 'R_value': r_value, 'R_squared': (r_value**2) if pd.notna(r_value) else np.nan})
    results_df = pd.DataFrame(results); slopes_df = pd.DataFrame(slopes)
    if not results_df.empty: results_df.insert(0, 'Station', station_name)
    if not slopes_df.empty: slopes_df.insert(0, 'Station', station_name)
    return results_df, slopes_df, station_name
print('Analyzer ready.')

Analyzer ready.


In [6]:
def batch_analyze_noresample(input_root: str, output_root: str):
    in_root = Path(input_root); out_root = Path(output_root)
    out_root.mkdir(parents=True, exist_ok=True)
    all_vol, all_slope = [], []
    for item in sorted(in_root.iterdir()):
        if not item.is_dir(): continue
        res_df, slp_df, st_name = analyze_flood_scaling_for_station_noresample(item)
        if res_df is None or res_df.empty:
            print(f"⚠️ 跳过：{item.name}（未找到事件文件或无有效数据）"); continue
        st_safe = re.sub(r"[^A-Za-z0-9_\-]+", "_", st_name if st_name else item.name)
        st_dir = out_root / st_safe; (st_dir / 'max_volume').mkdir(parents=True, exist_ok=True); (st_dir / 'scaling').mkdir(parents=True, exist_ok=True)
        vol_file = st_dir / 'max_volume' / f"{st_safe}_max_volume_results.csv"
        scl_file = st_dir / 'scaling'   / f"{st_safe}_scaling_results.csv"
        res_df.to_csv(vol_file, index=False); slp_df.to_csv(scl_file, index=False)
        print(f"✅ {st_name}: 保存 {vol_file.name} 与 {scl_file.name}")
        all_vol.append(res_df); all_slope.append(slp_df)
    if all_vol: pd.concat(all_vol, ignore_index=True).to_csv(out_root / 'ALL_max_volume_results.csv', index=False)
    if all_slope: pd.concat(all_slope, ignore_index=True).to_csv(out_root / 'ALL_scaling_results.csv', index=False)
    return {'output_root': str(out_root), 'n_stations_processed': len(all_vol)}
print('Batch ready.')

Batch ready.


In [7]:
result = batch_analyze_noresample(INPUT_ROOT, OUTPUT_ROOT)
result

✅ GARRAWAY CREEK JUNCTION: 保存 GARRAWAY_CREEK_JUNCTION_max_volume_results.csv 与 GARRAWAY_CREEK_JUNCTION_scaling_results.csv
✅ TELEGRAPH ROAD: 保存 TELEGRAPH_ROAD_max_volume_results.csv 与 TELEGRAPH_ROAD_scaling_results.csv
✅ SANDY CREEK: 保存 SANDY_CREEK_max_volume_results.csv 与 SANDY_CREEK_scaling_results.csv
✅ WAKOOKA ROAD: 保存 WAKOOKA_ROAD_max_volume_results.csv 与 WAKOOKA_ROAD_scaling_results.csv
✅ HRS BATTLE CAMP CROSSING: 保存 HRS_BATTLE_CAMP_CROSSING_max_volume_results.csv 与 HRS_BATTLE_CAMP_CROSSING_scaling_results.csv
✅ KALPOWAR CROSSING: 保存 KALPOWAR_CROSSING_max_volume_results.csv 与 KALPOWAR_CROSSING_scaling_results.csv
✅ HRS DEVELOPMENT ROAD: 保存 HRS_DEVELOPMENT_ROAD_max_volume_results.csv 与 HRS_DEVELOPMENT_ROAD_scaling_results.csv
✅ HRS BAIRDS: 保存 HRS_BAIRDS_max_volume_results.csv 与 HRS_BAIRDS_scaling_results.csv
✅ CHINA CAMP: 保存 CHINA_CAMP_max_volume_results.csv 与 CHINA_CAMP_scaling_results.csv
✅ MYOLA: 保存 MYOLA_max_volume_results.csv 与 MYOLA_scaling_results.csv
✅ KURANDA: 保存 KURANDA_

{'output_root': 'grdc_scaling_out', 'n_stations_processed': 658}

In [ ]:
from pathlib import Path; import pandas as pd
root = Path(OUTPUT_ROOT)
for nm in ['ALL_max_volume_results.csv', 'ALL_scaling_results.csv']:
    p = root / nm
    if p.exists():
        print(f"\n## {nm}")
        display(pd.read_csv(p).head(10))
    else:
        print(f"{nm} 未生成或无数据：{p}")